In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from mental_health_risk_prediction.preprocessing import IQRClipper


In [2]:
csv_path = "../data/raw/student-mental-health-6af1a2527d06c522087238.csv"

df = pd.read_csv(csv_path)
target_col = "Mental_Health_Risk"

In [3]:
df = df.drop(columns=["Student_ID"])
df = df.drop_duplicates().dropna(subset=[target_col])
print(df.shape)

(2000, 16)


In [4]:
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())

(1600, 15) (400, 15)
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64


In [5]:
numeric_cols = ["Age", "Year_of_Study", "Sleep_Hours", "Study_Hours_Per_Week", "GPA",
                "Social_Media_Hours", "Screen_Time_Hours", "Financial_Stress",
                "Family_Support", "Attendance_Rate"]
categorical_cols = ["Gender", "Physical_Activity", "Part_Time_Job",
                    "Diet_Quality", "Previous_Mental_Health_History"]

numeric_pipe = Pipeline([
    ("outliers", IQRClipper()),
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols),
], sparse_threshold=0.0)

In [6]:
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print(X_train_t.shape, X_test_t.shape)
print("NaN left:", pd.DataFrame(X_train_t).isna().sum().sum(), pd.DataFrame(X_test_t).isna().sum().sum())
pd.DataFrame(X_train_t, columns=preprocessor.get_feature_names_out()).head()

(1600, 23) (400, 23)
NaN left: 0 0


,num__Age,num__Year_of_Study,num__Sleep_Hours,num__Study_Hours_Per_Week,num__GPA,num__Social_Media_Hours,num__Screen_Time_Hours,num__Financial_Stress,num__Family_Support,num__Attendance_Rate,...,cat__Physical_Activity_High,cat__Physical_Activity_Low,cat__Physical_Activity_Moderate,cat__Part_Time_Job_No,cat__Part_Time_Job_Yes,cat__Diet_Quality_Average,cat__Diet_Quality_Good,cat__Diet_Quality_Poor,cat__Previous_Mental_Health_History_No,cat__Previous_Mental_Health_History_Yes
0,-0.063471,-0.471636,-1.407010,-0.239251,0.074785,0.150817,1.102483,0.713748,0.643008,-1.260010,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1,0.876838,-1.232340,-1.679538,-1.334412,-0.977908,0.331233,0.195167,-0.842691,0.368687,-0.473398,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,-1.003780,-0.471636,0.409842,-0.517387,-1.036391,-0.300224,-0.833123,-0.305988,0.752736,0.776753,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
3,0.876838,-1.232340,1.409111,-0.082800,0.893546,0.782274,1.102483,0.821088,-0.948052,-0.220559,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.406684,0.289067,0.409842,0.664691,-0.276113,-0.390432,-0.349222,0.660077,0.697872,-1.470710,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
